# EEEM068 — Knee MRI Classification using Vision Transformer

This notebook trains a Multi-View Vision Transformer (ViT-Small) on the MRNet dataset to classify three knee conditions: ACL tear, meniscus tear, and general abnormality.

**Model:** ViT-Small with DINO self-supervised pretrained weights  
**Total parameters:** 22,390,403  
**Dataset:** MRNet — 1,130 training scans, 120 validation scans


## Section 1 — Install and Imports

Install required packages and import all libraries.

- `timm` provides the ViT-Small architecture
- `kagglehub` downloads the MRNet dataset
- scikit-learn provides evaluation metrics


In [1]:
!pip install -q timm kagglehub

In [2]:
import os, gc, math, random, warnings, urllib.request
from pathlib import Path
from io import BytesIO
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, roc_auc_score, roc_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T
import timm

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"timm    : {timm.__version__}")


PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : NVIDIA RTX A4000
VRAM    : 16.7 GB
timm    : 1.0.26


## Section 2 — Configuration

All hyperparameters are in one `CFG` class so any value can be changed in one place.

**Key decisions:**

- `BATCH_SIZE = 8` — each forward pass loads 3 images (one per plane), so effective memory is 3x. Batch 8 fits on a 16GB GPU.
- `EPOCHS = 45`, `PATIENCE = 12`, `MIN_EPOCHS = 20` — train at least 20 epochs before early stopping triggers; stop after 12 epochs without improvement.
- `HEAD_LR = 3e-4` — classification head is randomly initialised so needs a higher learning rate.
- `BACKBONE_LR_LATE = 2e-5`, `BACKBONE_LR_EARLY = 5e-6` — pretrained backbone layers use much smaller rates to preserve learned features.
- `DROP_PATH_RATE = 0.1` — stochastic depth drops entire transformer blocks randomly during training to reduce co-adaptation.
- `DROPOUT_HEAD1 = 0.45`, `DROPOUT_HEAD2 = 0.20` — dropout in MLP head reduces overfitting on the small dataset.
- `MIXUP_ALPHA = 0.3` — MixUp blends two training samples. Starts at epoch 5 once backbone unfreezing begins.
- `FOCAL_GAMMA = 1.5`, `FOCAL_ALPHA = [0.77, 0.63, 0.20]` — Focal Loss parameters. Alpha = 1 - class_frequency so rarer classes get higher loss weight. ACL (23% positive) gets alpha=0.77; Abnormal (80%) gets alpha=0.20.


In [3]:
class CFG:
    OUT_DIR         = Path("./mrnet_outputs")
    CHECKPOINT_PATH = str(OUT_DIR / "best_vit.pth")
    TENSORBOARD_DIR = str(OUT_DIR / "tb_logs")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    IMAGE_SIZE  = 224
    MODEL_NAME  = "vit_small_patch16_224"
    NUM_CLASSES = 3
    LABEL_NAMES = ["acl", "meniscus", "abnormal"]

    TOP_K_SLICES = 5
    N_CHANNELS   = 3

    BATCH_SIZE  = 8
    EPOCHS      = 45
    MIN_EPOCHS  = 20
    PATIENCE    = 12
    NUM_WORKERS = 2
    SEED        = 42

    HEAD_LR           = 3e-4
    BACKBONE_LR_EARLY = 5e-6
    BACKBONE_LR_LATE  = 2e-5
    WEIGHT_DECAY      = 2e-4

    UNFREEZE_PARTIAL_EP = 4
    UNFREEZE_FULL_EP    = 8
    N_BLOCKS_PARTIAL    = 6

    WARMUP_EPOCHS  = 2
    WARMUP_RESTART = 2

    DROP_PATH_RATE = 0.1
    DROPOUT_HEAD1  = 0.45
    DROPOUT_HEAD2  = 0.20
    LABEL_SMOOTH   = 0.05
    MIXUP_ALPHA    = 0.3
    MIXUP_PROB     = 0.5
    MIXUP_START_EP = UNFREEZE_PARTIAL_EP + 1

    # alpha = 1 - label_frequency so rarer classes get higher loss weight
    # MRNet: ACL~23%, Meniscus~37%, Abnormal~80%
    FOCAL_GAMMA = 1.5
    FOCAL_ALPHA = [0.77, 0.63, 0.20]

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CFG loaded")
print(f"  Device       : {CFG.DEVICE}")
print(f"  Unfreeze     : ep{CFG.UNFREEZE_PARTIAL_EP} -> top-{CFG.N_BLOCKS_PARTIAL} blocks")
print(f"  Full unfreeze: ep{CFG.UNFREEZE_FULL_EP}")
print(f"  LR head={CFG.HEAD_LR}  bb_late={CFG.BACKBONE_LR_LATE}  bb_early={CFG.BACKBONE_LR_EARLY}")


CFG loaded
  Device       : cuda
  Unfreeze     : ep4 -> top-6 blocks
  Full unfreeze: ep8
  LR head=0.0003  bb_late=2e-05  bb_early=5e-06


## Section 3 — Dataset Download and Labels

The MRNet dataset (Bien et al., Stanford 2018) contains 1,370 knee MRI exams. Each exam has three 3D volumes stored as `.npy` files, one per imaging plane (sagittal, coronal, axial).

Labels are provided as separate CSV files per condition. `load_labels` merges all three and creates a multi-label target vector `[acl, meniscus, abnormal]` per exam.

**Class distribution in training set:**
- ACL: 208 positive out of 1130 (18%) — most imbalanced
- Meniscus: 397 positive (35%)
- Abnormal: 913 positive (81%) — majority class

For Colab: uncomment the Drive mount block and comment out the local setup lines.


In [4]:
# Google Colab setup — uncomment if running on Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# os.makedirs('/root/.config/kaggle', exist_ok=True)
# os.system('cp /content/drive/MyDrive/kaggle.json /root/.config/kaggle/kaggle.json')
# os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# Local setup:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.system("cp ~/Downloads/kaggle.json ~/.kaggle/kaggle.json")
os.system("chmod 600 ~/.kaggle/kaggle.json")

import kagglehub
dl_path = kagglehub.dataset_download("cjinny/mrnet-v1")
CFG.ROOT = Path(dl_path) / "MRNet-v1.0"
assert CFG.ROOT.exists(), f"Dataset not found: {CFG.ROOT}"
print(f"Dataset at: {CFG.ROOT}")


cp: cannot stat '/user/HS402/mi00806/Downloads/kaggle.json': No such file or directory
chmod: cannot access '/user/HS402/mi00806/.kaggle/kaggle.json': No such file or directory


Dataset at: /user/HS402/mi00806/.cache/kagglehub/datasets/cjinny/mrnet-v1/versions/1/MRNet-v1.0


In [5]:
def load_labels(root, split):
    acl      = pd.read_csv(root / f"{split}-acl.csv",      header=None, names=["id", "acl"])
    abnormal = pd.read_csv(root / f"{split}-abnormal.csv", header=None, names=["id", "abnormal"])
    meniscus = pd.read_csv(root / f"{split}-meniscus.csv", header=None, names=["id", "meniscus"])
    df = acl.merge(abnormal, on="id").merge(meniscus, on="id")
    df["id"] = df["id"].astype(str).str.zfill(4)
    for plane in ["sagittal", "coronal", "axial"]:
        df[f"{plane}_path"] = df["id"].apply(
            lambda x: str(root / split / plane / f"{x}.npy"))
    df["target"] = df.apply(
        lambda r: [float(r["acl"]), float(r["meniscus"]), float(r["abnormal"])], axis=1)
    return df

train_df = load_labels(CFG.ROOT, "train")
valid_df = load_labels(CFG.ROOT, "valid")
print(f"Train: {len(train_df)}  Valid: {len(valid_df)}")
for lbl in CFG.LABEL_NAMES:
    n = int(train_df[lbl].sum())
    print(f"  {lbl:>10}: {n} pos ({100*n/len(train_df):.1f}%)  {len(train_df)-n} neg")


Train: 1130  Valid: 120
         acl: 208 pos (18.4%)  922 neg
    meniscus: 397 pos (35.1%)  733 neg
    abnormal: 913 pos (80.8%)  217 neg
